[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/01_tokenization/01_bpe_from_scratch.ipynb)

# 模块 01 · Tokenization：从零实现 BPE

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 Python，无任何依赖（matplotlib 仅用于一张图，tiktoken 可选）。

**本 notebook 你将完成：**

1. 在一段 ~2KB 英文语料上**从零训练 byte-level BPE**（`get_stats` → `merge` → 训练循环学 50 个 merges）；
2. 实现 `encode` / `decode` 并验证 round-trip 无损；
3. 量化**压缩率**随词表增长的变化（vocab 256 → 306）；
4. 与 **tiktoken (gpt2)** 对照切分，并观察**数字切分**的混乱；
5. 3 道 ✏️ 练习巩固核心函数。

参考：[Sennrich 2015] *Neural Machine Translation of Rare Words with Subword Units* (arXiv:1508.07909)、[Kudo 2018] *SentencePiece* (arXiv:1808.06226)、[Karpathy minbpe] (github.com/karpathy/minbpe)。本实现与 minbpe 的 `BasicTokenizer` 同构。

In [ ]:
# 内嵌训练语料（~2KB 英文）。byte-level BPE：先 UTF-8 编码成字节流，初始词表 = 256 个字节值。
corpus = '''Language models do not read text the way people do. Before a single weight is touched, \
the text is split into tokens, and the model only ever sees the integer ids of those tokens. \
The tokenizer is therefore the interface between the raw bytes of the world and the neural \
network that must make sense of them. If the tokenizer throws information away, the model \
can never get it back.

The byte pair encoding algorithm starts from the smallest possible units and then merges the \
most frequent pair of neighbouring units, over and over again. The most common pairs in \
English text are easy to guess: the letter t followed by the letter h, the letter e followed \
by a space, and the word the surrounded by spaces. After a few hundred merges the tokenizer \
has learned the most common words of the language, and after a few thousand merges it has \
learned most of the everyday vocabulary.

The beauty of the method is that it never fails. When the tokenizer meets a word it has never \
seen, it simply falls back to smaller pieces, and in the worst case it falls all the way back \
to single bytes. There is no unknown token, no lost text, and no special case in the code. \
The same loop that compresses the most common words also handles the rarest ones.

The cost of the method is that the splits follow frequency and not meaning. The number 12345 \
may be split in one way and the number 12346 in another, and the model has to learn arithmetic \
across these arbitrary boundaries. The same sentence written in another language can cost two \
or three times as many tokens, which means less context and more money for the same content. \
The tokenizer shapes what the model finds easy and what the model finds hard, and that is why \
we study it first.

None of this is hard to fix in isolation, and none of it is free to fix at scale. Every change \
to the tokenizer invalidates the embedding table of the trained model, so the choices made on \
day one tend to follow a model family for years. Measure twice, merge once.'''

byte_ids = list(corpus.encode("utf-8"))   # 字节流：每个元素是 0..255 的 int
print(f"语料字符数 : {len(corpus)}")
print(f"UTF-8 字节数: {len(byte_ids)}")
print(f"前 40 个字节: {byte_ids[:40]}")
print(f"对应文本    : {corpus[:40]!r}")

## 1 · BPE 训练：`get_stats` → `merge` → 循环

训练循环只有三步（[Sennrich 2015]）：

1. **`get_stats(ids)`**：统计序列中所有**相邻 pair** `(ids[i], ids[i+1])` 的出现次数；
2. 取频率最高的 pair，分配一个新 token id（从 256 开始递增）；
3. **`merge(ids, pair, new_id)`**：把序列中所有该 pair 替换成 `new_id`，回到第 1 步。

重复 $k$ 次，词表从 256 涨到 $256+k$。每条 merge 规则 `(a, b) -> new_id` 按学习顺序存进 `merges` 字典——编码新文本时要**按同样顺序**重放。

In [ ]:
def get_stats(ids):
    # 统计相邻 pair 频率：ids=[1,2,3,1,2] -> {(1,2):2, (2,3):1, (3,1):1}
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    # 把 ids 中所有相邻的 pair 替换为 new_id（从左到右，不重叠）
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

# sanity check
assert get_stats([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert merge([5, 1, 2, 9, 1, 2], (1, 2), 256) == [5, 256, 9, 256]
print("get_stats / merge OK")

In [ ]:
NUM_MERGES = 50               # 学 50 条合并规则：vocab 256 -> 306

ids = list(byte_ids)          # 不破坏原始字节流
merges = {}                   # (a, b) -> new_id，dict 保持插入顺序 = 训练顺序
vocab = {i: bytes([i]) for i in range(256)}   # token id -> 原始字节串
seq_lens = [len(ids)]         # 记录每一步的序列长度，给压缩率曲线用

for i in range(NUM_MERGES):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)          # 频率最高的相邻 pair
    new_id = 256 + i
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
    seq_lens.append(len(ids))
    if i < 10:                                # 打印前 10 条学到的规则
        print(f"merge {i+1:2d}: {pair} -> {new_id}   {vocab[new_id]!r:12}  freq={stats[pair]}")

print(f"\n训练完成：{len(byte_ids)} bytes -> {len(ids)} tokens，"
      f"压缩率 {len(byte_ids)/len(ids):.2f}x")

注意前几条规则：`'e '`、`' th'`、`' the '` 这类英文最高频组合**最先**被合并出来——BPE 学到的就是语料的频率统计，第 4 条规则 `' the '` 就已经把英语最高频词合成了独立 token。

## 2 · `encode` / `decode` 与 round-trip 验证

- **`encode(text)`**：文本 → 字节流 → 反复找出"当前序列中所有可合并 pair 里，**训练序号最小**的那条规则"并应用，直到没有规则可用。必须按训练顺序，因为后学的规则依赖先学的（minbpe 同款实现）。
- **`decode(ids)`**：查 `vocab` 拼回字节串，再 UTF-8 解码。`errors="replace"` 防御单个 token 停在多字节字符中间的情况（整段解码时字节是完整的，所以 round-trip 仍然无损）。

In [ ]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # 在当前所有相邻 pair 中，找训练序号（new_id）最小的可合并规则
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                      # 没有任何 pair 在规则表里 -> 结束
        ids = merge(ids, pair, merges[pair])
    return ids

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

# round-trip：编码再解码必须逐字符还原
for s in [corpus, "hello world", "the theory of the throne",
          "BPE never fails: 中文 / emoji 🙂 / x99!"]:
    assert decode(encode(s)) == s
print("round-trip 全部通过")

demo = "the tokenizer never fails"
print(f"\n{demo!r} -> {encode(demo)}")
print("逐 token 还原:", [decode([t]) for t in encode(demo)])

中文和 emoji 在训练语料里没出现过，但 round-trip 依然无损——它们退化成逐字节 token（一个汉字 3 个 token、emoji 4 个 token）。**无 OOV** 是 byte-level 的结构性保证，代价是非拉丁文字的"token 税"。

## 3 · 压缩率分析：vocab 256 → 306

训练时记录的 `seq_lens` 就是答案：每学一条 merge，序列就变短一点。画出 token 数随词表大小的变化。

In [ ]:
import matplotlib.pyplot as plt

vocab_sizes = list(range(256, 256 + NUM_MERGES + 1))
plt.figure(figsize=(7, 4))
plt.plot(vocab_sizes, seq_lens, marker=".", ms=4)
plt.xlabel("vocab size")
plt.ylabel("sequence length (tokens)")
plt.title("BPE compression: more merges -> shorter sequence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"vocab 256: {seq_lens[0]} tokens   (1.00x)")
print(f"vocab 306: {seq_lens[-1]} tokens   ({seq_lens[0]/seq_lens[-1]:.2f}x)")
print("注意曲线斜率递减：高频 pair 先被合并，收益最大；越往后新 merge 越偏门，边际收益越小。")
print("这正是『词表大小 scaling』权衡的微观来源（讲解第 7 节）。")

## 4 · 与 tiktoken 对照 + 数字切分

我们的 50-merge 玩具 tokenizer vs GPT-2 的 50257 词表（tiktoken 未安装则跳过，不影响后续 cell）。再看 `"12345+67890"`：BPE 的数字切分边界与数位毫无关系——这是 LLM 算术弱的感知层原因之一。

In [ ]:
sentence = "The tokenizer is the interface between the bytes and the model."
ours = encode(sentence)
print(f"句子: {sentence!r}")
print(f"我们的 BPE (vocab 306) : {len(ours):3d} tokens  {[decode([t]) for t in ours][:12]} ...")

try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    tk = enc.encode(sentence)
    print(f"tiktoken gpt2 (50257)  : {len(tk):3d} tokens  {[enc.decode([t]) for t in tk]}")
    print("\n观察: gpt2 的 token 大多以空格开头（' the'）——预切分把空格归给后面的词；")
    print("词表大 160 倍，压缩率也高得多。")
except ImportError:
    print("tiktoken 未安装（pip install tiktoken 可选），跳过对照。")

In [ ]:
expr = "12345+67890"
print(f"表达式: {expr!r}\n")
ours = encode(expr)
print(f"我们的 BPE : {[decode([t]) for t in ours]}")
print("  -> 训练语料里数字极少，每个数字各占 1 字节 token，边界尚且规整。\n")

try:
    import tiktoken
    for name in ["gpt2", "cl100k_base"]:
        enc = tiktoken.get_encoding(name)
        tk = enc.encode(expr)
        print(f"{name:12s}: {[enc.decode([t]) for t in tk]}")
    print("\ngpt2 把 '12345' 切成与数位无关的碎片；cl100k (GPT-4) 强制 1-3 位分组，")
    print("但 '12345' -> '123'+'45' 依然和个十百千对不齐。模型做加法前得先『反解』")
    print("每个 token 覆盖了哪几位——tokenization 直接塑造了模型觉得什么难、什么容易。")
except ImportError:
    print("tiktoken 未安装，给出 gpt2 的实际切分供参考: ['123', '45', '+', '678', '90']")

---
## ✏️ 练习 1：实现 `get_stats_ex`

不翻上文，自己实现一遍 pair 频率统计：输入 int 列表 `ids`，返回 `{(a, b): count}`，统计所有**相邻**二元组的出现次数。

**提示**：`zip(ids, ids[1:])` 一行就能产生所有相邻 pair；空列表和单元素列表应返回 `{}`。10 行以内。

In [ ]:
def get_stats_ex(ids):
    # TODO: 统计 ids 中所有相邻 pair 的出现次数，返回 dict {(a, b): count}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert get_stats_ex([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert get_stats_ex([5, 5, 5]) == {(5, 5): 2}      # 重叠 pair 也要数：位置 (0,1) 和 (1,2)
assert get_stats_ex([]) == {}                      # 边界：空序列
assert get_stats_ex([7]) == {}                     # 边界：单元素无 pair
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `merge_ex`

实现合并：把 `ids` 中所有相邻的 `pair` 替换为 `new_id`，**从左到右扫描、不重叠**。

**提示**：`while i < len(ids)` + 命中则 `i += 2`、未命中则 `i += 1`。关键边界：`[1,1,1]` 合并 `(1,1)` 时，位置 0-1 先命中消耗掉，位置 1-2 的重叠 pair 不再匹配，结果是 `[9, 1]` 而不是 `[1, 9]` 或 `[9, 9]`。15 行以内。

In [ ]:
def merge_ex(ids, pair, new_id):
    # TODO: 从左到右扫描，把所有相邻的 pair 替换为 new_id（不重叠）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert merge_ex([1, 2, 3], (1, 2), 9) == [9, 3]
assert merge_ex([1, 1, 1], (1, 1), 9) == [9, 1]            # 连续重叠：左侧优先
assert merge_ex([1, 1, 1, 1], (1, 1), 9) == [9, 9]
assert merge_ex([5, 1, 1, 2, 1, 1], (1, 1), 9) == [5, 9, 2, 9]
assert merge_ex([3, 4], (1, 2), 9) == [3, 4]               # 无命中：原样返回
assert merge_ex([], (1, 2), 9) == []                       # 边界：空序列
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `vocab_size_vs_seq_len`

把第 3 节的压缩率分析封装成函数：`vocab_size_vs_seq_len(text, n_merges_list)` 对每个 `n`，在 `text` 上从零训练 `n` 个 merges，返回压缩率表——`[(n_merges, seq_len, compression_ratio), ...]`，其中 `compression_ratio = 字节数 / seq_len`。

**提示**：直接复用上面的 `get_stats` 和 `merge`；每个 `n` 都从原始字节流重新训练（互不影响）；若 `stats` 为空（序列太短合并完了）提前 `break`。约 15 行。

In [ ]:
def vocab_size_vs_seq_len(text, n_merges_list):
    # TODO: 对每个 n ∈ n_merges_list：
    #   1) ids = 原始 UTF-8 字节流
    #   2) 训练 n 步：每步取频率最高 pair，合并为新 id（256+i）；stats 为空则 break
    #   3) 记录 (n, len(ids), 字节数/len(ids))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
sample = corpus[:600]
n_bytes = len(sample.encode("utf-8"))
table = vocab_size_vs_seq_len(sample, [0, 5, 15, 30])

assert len(table) == 4
assert table[0][1] == n_bytes                                  # 0 merges -> 长度 = 字节数
lens = [row[1] for row in table]
assert all(lens[i] >= lens[i + 1] for i in range(len(lens) - 1))   # 单调：merges 越多序列越短
assert lens[0] > lens[-1]                                      # 30 merges 必须真的压短了
for n, L, r in table:
    assert abs(r - n_bytes / L) < 1e-9                         # 压缩率口径正确
print("✅ 练习 3 通过")
for n, L, r in table:
    print(f"  merges={n:3d}  vocab={256+n}  seq_len={L:4d}  压缩率={r:.3f}x")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def get_stats_ex(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def merge_ex(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def vocab_size_vs_seq_len(text, n_merges_list):
    base = list(text.encode("utf-8"))
    n_bytes = len(base)
    results = []
    for n in n_merges_list:
        ids = list(base)
        for i in range(n):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            ids = merge(ids, pair, 256 + i)
        results.append((n, len(ids), n_bytes / len(ids)))
    return results

---
## 小结

- BPE 训练 = `get_stats`（统计相邻 pair）+ `merge`（合并最频 pair）+ 循环 $k$ 次；编码 = 按训练顺序重放 merges。整个算法 ~30 行 Python（[Karpathy minbpe] 同构）。
- byte-level 起步（256 字节）保证**任何字符串可编码、round-trip 无损、无 OOV**；代价是未被合并覆盖的语言/符号付出更高 token 税。
- 压缩率随词表增长**边际递减**——词表大小是 scaling 权衡而非越大越好。
- 数字切分与数位无关、前导空格改变 token 身份、跨 tokenizer 的 PPL/logprob 不可直接比——评测前先审 tokenizer。

**下一步 → 模块 02 · 手写 Attention 与 Transformer Block**：token id 进入模型后变成 embedding 向量，注意力机制让它们互相交换信息。我们将从一个 `(B, T, C)` 张量出发，徒手搭出完整的 Transformer Block。

---
## 🎯 真实数据胶囊题：在真实莎士比亚文本上量化 BPE 压缩率

BPE 的价值是压缩：把字节序列合并成更少的 token。下载真实 tiny-shakespeare，实现 `compression_ratio`，衡量“做了 N 次最频繁 bigram 合并后，序列变短了多少”。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def shakespeare():
    return open(_fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

txt = shakespeare()[:20000]   # 取一段真实文本
ids = list(txt.encode("utf-8"))
print(f"真实文本 {len(txt)} 字符 -> {len(ids)} 字节")

def most_common_pair(ids):
    from collections import Counter
    c=Counter(zip(ids, ids[1:]))
    return c.most_common(1)[0][0] if c else None
def merge(ids, pair, new_id):
    out=[]; i=0
    while i<len(ids):
        if i<len(ids)-1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
            out.append(new_id); i+=2
        else: out.append(ids[i]); i+=1
    return out
print("worked: 一次最频繁合并", most_common_pair(ids))

**练习**：实现 `bpe_compress(ids, n_merges)`：做 `n_merges` 次“最频繁 pair 合并”，返回压缩后的 id 列表。再实现 `compression_ratio(orig, compressed)` = 原长/压缩后长。

In [ ]:
def bpe_compress(ids, n_merges):
    # TODO: 循环 n_merges 次：找最频繁 pair -> merge，new_id 从 256 起递增
    raise NotImplementedError
def compression_ratio(orig, compressed):
    # TODO: len(orig)/len(compressed)
    raise NotImplementedError


In [ ]:
# 自测
comp = bpe_compress(ids, 50)
r = compression_ratio(ids, comp)
assert len(comp) < len(ids), "合并后应更短"
assert r > 1.0
assert compression_ratio(ids, ids)==1.0
print(f"真实文本 50 次合并压缩率 = {r:.2f}x ✓")


### 📖 参考答案

In [ ]:
def bpe_compress(ids, n_merges):
    ids=list(ids); nid=256
    for _ in range(n_merges):
        p=most_common_pair(ids)
        if p is None: break
        ids=merge(ids,p,nid); nid+=1
    return ids
def compression_ratio(orig, compressed): return len(orig)/len(compressed)
print("✓ BPE 在真实文本上的压缩率就是它存在的理由")